In [1]:
from glmsingle.glmsingle import GLM_single
import argparse
import os
import os.path as op
from nilearn import image
from numrisk.utils.data import Subject
from nilearn.glm.first_level import make_first_level_design_matrix
import numpy as np
import pandas as pd
import nibabel as nib

import warnings
warnings.filterwarnings('ignore')

In [2]:
TR = 2.827
stim_duration = 0.6 
subject = '01'
bids_folder = '/mnt_AdaBD_largefiles/Data/SMILE_DATA/DNumRisk/ds-numrisk'
space = 'T1w'
runs = range(1, 7) 
session = 1
task='magjudge'
coOccCV=True
perstim=True
print('hey')

hey


In [3]:
def get_fmri_events_bothStim_coOccCV_perstim(sub, session, runs, bids_folder): # grouping stim 1 and stim 2 as conditions, not numerosities
    behavior = []
    for run in runs:
        behavior.append(pd.read_table(op.join(
            bids_folder, f'sub-{sub}/ses-{session}/func/sub-{sub}_ses-{session}_task-magjudge_run-{run}_events.tsv')))

    behavior = pd.concat(behavior, keys=runs, names=['run'])
    behavior = behavior.reset_index().set_index(
        ['run', 'trial_type'])

    behavior = behavior[behavior['trial_nr'] != 0]

    stimulus1 = behavior.xs('stimulus 1', 0, 'trial_type', drop_level=False).reset_index('trial_type')[['onset', 'trial_nr', 'trial_type', 'n1']]
    stimulus1['duration'] = stim_duration
    stimulus1['trial_type'] = stimulus1.n1.map(lambda n1: f'n1')

    stimulus2 = behavior.xs('stimulus 2', 0, 'trial_type', drop_level=False).reset_index('trial_type')[['onset', 'trial_nr', 'trial_type', 'n2']]
    stimulus2['duration'] = stim_duration
    stimulus2['trial_type'] = stimulus2.n2.map(lambda n2: f'n2')

    events = pd.concat((stimulus1, stimulus2)).sort_index()
    events = events[['onset', 'duration', 'trial_type']]  
    
    return events 

In [5]:
# not there yet, also saving output step has to be adapted
def load_fmri_data(subject,bids_folder, space,session=1, task = 'magjudge', runs=range(1, 7)):
    """Load fMRI data from BIDS derivatives (supports NIfTI and GIFTI surface data)."""
    import nibabel as nib
    base = op.join(bids_folder, 'derivatives', 'fmriprep',f'sub-{subject}', f'ses-{session}', 'func')

    data = []
    for run in runs:
        if "fsaverage" in space:  # Surface data
            hemi_data = []
            for hemi in ['L', 'R']:
                path = op.join(base, f'sub-{subject}_ses-{session}_task-{task}_run-{run}_space-{space}_hemi-{hemi}_bold.func.gii')
                gii = nib.load(path)
                hemi_data.append(np.column_stack([d.data for d in gii.darrays]))
            data.append(np.vstack(hemi_data))  # Combine hemispheres
        else:  # Volume data
            path = op.join(base, f'sub-{subject}_ses-{session}_task-{task}_run-{run}_space-{space}_desc-preproc_bold.nii.gz')
            data.append(nib.load(path).get_fdata())
    return data 

In [7]:
    
derivatives = op.join(bids_folder, 'derivatives')
subject = f'{int(subject):02d}'

key = f'glm_stim.denoise'
key += '.coOccCV'  if coOccCV else ''
key += '.perstim' if perstim else ''

base_dir = op.join(derivatives, key, f'sub-{subject}', f'ses-{session}', 'func')
#os.makedirs(base_dir, exist_ok=True) # will be created by GLMsingle!

# get fMRI data
im_data = load_fmri_data(subject, bids_folder=bids_folder, space=space) # _bold missing for numrisk

onsets = get_fmri_events_bothStim_coOccCV_perstim(subject, session, runs, bids_folder)
print("Using co-occurring cross-validation design matrix for both stimuli, per stimulusnot per numerosity.")

tr = TR
N_volumes = np.shape(im_data)[-1] # number of volumes
frametimes = np.linspace(tr/2., (N_volumes - .5)*tr, N_volumes)
onsets['onset'] = ((onsets['onset']+tr/2.) // tr) * tr
dm = [make_first_level_design_matrix(frametimes, onsets.loc[run], hrf_model='fir', oversampling=100.,
                                        drift_order=0,
                                        drift_model=None).drop('constant', axis=1) for run in runs]
dm = pd.concat(dm, keys=runs, names=['run']).fillna(0) # keys = range(1, 7)
dm.columns = [c.replace('_delay_0', '') for c in dm.columns]
dm /= dm.max()
dm[dm < 1.0] = 0.0
X = [dm.loc[run].values for run in runs]

print("Design matrix and data shapes:")
print(np.shape(X))
print(np.shape(im_data))

# set options for GLM-single
opt = dict()
opt['wantlibrary'] = 1 # set important fields for completeness (but these would be enabled by default)
opt['wantglmdenoise'] = 1
opt['wantfracridge'] = 1
opt['wantfileoutputs'] = [0, 0, 0, 1] # keep the relevant outputs in memory and also save them to the disk

glmsingle_obj = GLM_single(opt)
results_glmsingle = glmsingle_obj.fit(
    X,
    im_data,
    stim_duration,
    tr,
    outputdir=base_dir,
    figuredir = op.join(base_dir, 'GLMestimatesingletrialfigures') # would be written to cwd otherwise and could crash when multiple nodes use it a the same time 
    )

# Save results: separate n1 and n2 betas
betas = results_glmsingle['typed']['betasmd']
if space == 'T1w':
    base = op.join(bids_folder, 'derivatives', 'fmriprep',f'sub-{subject}', f'ses-{session}', 'func')
    example_image = op.join(base, f'sub-{subject}_ses-{session}_task-{task}_run-1_space-{space}_desc-preproc_bold.nii.gz')
    betas = image.new_img_like(example_image, betas)

    # n1s
    betas_n1 = image.index_img(betas, slice(0, None, 2) ) # slice(0, None, 2) where to start, where to end, step size
    #betas_n1.to_filename(op.join(base_dir, f'sub-{subject}_ses-{session}_task-magjudge_space-{space}_desc-stims1_pe.nii.gz'))
    # n2s
    betas_n2 = image.index_img(betas, slice(1, None, 2) ) # slice(0, None, 2) where to start, where to end, step size
    #betas_n2.to_filename(op.join(base_dir, f'sub-{subject}_ses-{session}_task-magjudge_space-{space}_desc-stims2_pe.nii.gz'))



Using co-occurring cross-validation design matrix for both stimuli, per stimulusnot per numerosity.
Design matrix and data shapes:
(6, 189, 2)
(6, 54, 66, 50, 189)
*** DIAGNOSTICS ***:
There are 6 runs.
The number of conditions in this experiment is 2.
The stimulus duration corresponding to each trial is 0.60 seconds.
The TR (time between successive data points) is 2.83 seconds.
The number of trials in each run is: [59, 60, 59, 59, 59, 59].
The number of trials for each condition is: [180, 175].
For each condition, the number of runs in which it appears: [6, 6].
For each run, how much ending buffer do we have in seconds? [113.08, 107.426, 113.08, 113.08, 110.253, 113.08].
*** Saving design-related results to /mnt_AdaBD_largefiles/Data/SMILE_DATA/DNumRisk/ds-numrisk/derivatives/glm_stim.denoise.coOccCV.perstim/sub-01/ses-1/func/DESIGNINFO.npy. ***
*** FITTING DIAGNOSTIC RUN-WISE FIR MODEL ***
*** Saving FIR results to /mnt_AdaBD_largefiles/Data/SMILE_DATA/DNumRisk/ds-numrisk/derivatives

chunks: 100%|██████████| 4/4 [03:56<00:00, 59.06s/it]


*** DETERMINING GLMDENOISE REGRESSORS ***

*** CROSS-VALIDATING DIFFERENT NUMBERS OF REGRESSORS ***



chunks: 100%|██████████| 4/4 [13:02<00:00, 195.61s/it]



*** FITTING TYPE-C MODEL (GLMDENOISE) ***



chunks: 100%|██████████| 4/4 [01:43<00:00, 25.88s/it]


*** FITTING TYPE-D MODEL (GLMDENOISE_RR) ***



chunks: 100%|██████████| 4/4 [30:47<00:00, 461.96s/it]



*** Saving results to /mnt_AdaBD_largefiles/Data/SMILE_DATA/DNumRisk/ds-numrisk/derivatives/glm_stim.denoise.coOccCV.perstim/sub-01/ses-1/func/TYPED_FITHRF_GLMDENOISE_RR.npy. ***

*** All model types done ***

*** return model types in results ***

